# 第2回発展：用語集データをLoRAで学習し、学習前後を比較

`lectures/02/glossary_sft.jsonl` を使い、Qwen/Qwen3.5-0.8B にLoRAを適用します。
評価は、用語集にある質問、言い換え、用語集にない質問を分けて、形式・内容・未知質問への確認要求を学習前後で比較します。

In [ ]:
import subprocess
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout
print(gpu_info or 'GPUが見つかりません。ランタイムをT4 GPUに変更してください。')

In [ ]:
PINS = {'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0'}
spec = ' '.join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao
%pip install -q {spec}
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

In [ ]:
from pathlib import Path
import json, re, time, random
CONFIG = {
    'base_model': 'Qwen/Qwen3.5-0.8B',
    'revision': '2fc06364715b967f1860aea9cf38778875588b17',
    'seed': 42, 'num_train_epochs': 20, 'warmup_steps': 12,
    'lr': 2e-4, 'lora_r': 16, 'lora_alpha': 32,
    'max_seq_len': 512, 'max_new_tokens': 128,
    'work_dir': '/content/glossary_lora_lab'
}
WORK = Path(CONFIG['work_dir']); WORK.mkdir(parents=True, exist_ok=True)
random.seed(CONFIG['seed']); torch.manual_seed(CONFIG['seed'])
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## データの読み込み

Colabのファイル一覧に `glossary_sft.jsonl`、`glossary_eval.json`、`glossary.json` をアップロードしてから実行します。

In [ ]:
DATA_PATH = Path('/content/glossary_sft.jsonl')
EVAL_PATH = Path('/content/glossary_eval.json')
GLOSSARY_PATH = Path('/content/glossary.json')
if not (DATA_PATH.exists() and EVAL_PATH.exists() and GLOSSARY_PATH.exists()):
    from google.colab import files
    print('glossary_sft.jsonl / glossary_eval.json / glossary.json を選択してください')
    files.upload()
assert DATA_PATH.exists(), f'{DATA_PATH} がありません'
assert EVAL_PATH.exists(), f'{EVAL_PATH} がありません'
assert GLOSSARY_PATH.exists(), f'{GLOSSARY_PATH} がありません'
records = [json.loads(line) for line in DATA_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
glossary = json.loads(GLOSSARY_PATH.read_text(encoding='utf-8'))
terms = {x['id']: x for x in glossary['terms']}
eval_spec = json.loads(EVAL_PATH.read_text(encoding='utf-8'))
assert len(records) > 0 and all(len(x['messages']) == 2 for x in records)
print('SFT records:', len(records), '| terms:', len(terms), '| eval:', sum(len(eval_spec[k]) for k in ['known','paraphrase','unknown']))

In [ ]:
SYSTEM = ('あなたは講座の用語集回答アシスタントです。用語集に根拠がある場合は簡潔に説明し、'
          '根拠がない質問には推測で答えず、用語集に記載がなく確認が必要だと伝えてください。'
          '必ずJSON 1行で返してください。形式は {\"answer\": 文字列, \"needs_review\": 真偽値} です。')

def target(answer, needs_review=False):
    return json.dumps({'answer': answer, 'needs_review': needs_review}, ensure_ascii=False)

# 用語ごとに最初の質問を学習、2つ目を検証に回す。質問文の完全一致を避けて評価する。
train, valid = [], []
for i, record in enumerate(records):
    q = record['messages'][0]['content']; answer = record['messages'][1]['content']
    row = {'question': q, 'target': target(answer)}
    (train if i % 2 == 0 else valid).append(row)
assert len(train) == len(valid)
print(f'train={len(train)} valid={len(valid)}')
print(train[0])

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(CONFIG['base_model'], revision=CONFIG['revision'])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG['base_model'], revision=CONFIG['revision'], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print('loaded:', CONFIG['base_model'], '| device:', DEVICE, '| memory(GB):', BASE_MEMORY_GB)

In [ ]:
def build_messages(question):
    return [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': question}]

def generate(current_model, question):
    prompt = tok.apply_chat_template(build_messages(question), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors='pt').to(current_model.device)
    with torch.no_grad():
        out = current_model.generate(**inputs, max_new_tokens=CONFIG['max_new_tokens'], pad_token_id=tok.pad_token_id, do_sample=False)
    return tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def check_format(raw):
    try: obj = json.loads(raw)
    except Exception: return False, None, 'json_error'
    if not isinstance(obj, dict) or set(obj) != {'answer', 'needs_review'}: return False, obj, 'schema'
    if not isinstance(obj['answer'], str) or not isinstance(obj['needs_review'], bool): return False, obj, 'type'
    return True, obj, ''

KEYWORDS = {'lora': ['低ランク', 'PEFT'], 'qlora': ['4ビット', '量子化'], 'ai-agent': ['ツール', 'ループ'], 'pydantic-ai': ['Python', 'エージェント'], 'mcp': ['プロトコル', 'ツール']}
eval_rows = []
for group in ['known', 'paraphrase', 'unknown']:
    for item in eval_spec[group]:
        row = dict(item); row['group'] = group; eval_rows.append(row)
for row in eval_rows:
    if row['needs_review']:
        row['must_include'] = ['記載', '確認']
    else:
        row['must_include'] = KEYWORDS[row['term_id']]

def evaluate(current_model, label):
    rows = []; t0 = time.time()
    for item in eval_rows:
        raw = generate(current_model, item['question'])
        ok, obj, reason = check_format(raw)
        content_ok = bool(ok and obj['needs_review'] == item['needs_review'] and all(k in obj['answer'] for k in item['must_include']))
        rows.append({**item, 'raw': raw, 'format_ok': ok, 'format_reason': reason, 'content_ok': content_ok, 'both_ok': ok and content_ok})
    n = len(rows)
    summary = {'label': label, 'n': n, 'format_rate': sum(x['format_ok'] for x in rows)/n, 'content_rate': sum(x['content_ok'] for x in rows)/n, 'both_rate': sum(x['both_ok'] for x in rows)/n, 'seconds': round(time.time()-t0, 1)}
    (WORK / f'eval_{label}.json').write_text(json.dumps({'summary': summary, 'rows': rows}, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

In [ ]:
before_summary, before_rows = evaluate(model, 'before')
import pandas as pd
display(pd.DataFrame([before_summary]))
display(pd.DataFrame([{'group': r['group'], 'question': r['question'], 'raw': r['raw'], 'format/content': f"{r['format_ok']}/{r['content_ok']}"} for r in before_rows]))

## LoRA学習

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
def to_chat(row):
    return {'messages': build_messages(row['question']) + [{'role': 'assistant', 'content': row['target'] }]}
train_ds = Dataset.from_list([to_chat(x) for x in train])
valid_ds = Dataset.from_list([to_chat(x) for x in valid])
model.config.use_cache = False
lora = LoraConfig(r=CONFIG['lora_r'], lora_alpha=CONFIG['lora_alpha'], lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
sft = SFTConfig(output_dir=str(WORK/'ckpt'), num_train_epochs=CONFIG['num_train_epochs'], per_device_train_batch_size=4, gradient_accumulation_steps=2, learning_rate=CONFIG['lr'], lr_scheduler_type='cosine', warmup_steps=CONFIG['warmup_steps'], logging_steps=10, save_steps=50, save_total_limit=2, fp16=False, max_length=CONFIG['max_seq_len'], packing=False, report_to='none', seed=CONFIG['seed'])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time(); trainer.train()
print(f'train_seconds={time.time()-t0:.1f} steps={trainer.state.global_step} epoch={trainer.state.epoch:.1f}')
ADAPTER_DIR = WORK/'adapter'; trainer.model.save_pretrained(ADAPTER_DIR)
tuned = trainer.model; tuned.config.use_cache = True; tuned.eval()

In [ ]:
after_summary, after_rows = evaluate(tuned, 'after')
comparison = pd.DataFrame([before_summary, after_summary]).set_index('label')[['n','format_rate','content_rate','both_rate','seconds']]
display(comparison)
regressed = [b['question'] for b, a in zip(before_rows, after_rows) if b['both_ok'] and not a['both_ok']]
print('regressed:', regressed)
detail = pd.DataFrame([{'group': b['group'], 'question': b['question'], 'before': b['raw'], 'after': a['raw'], 'before_format/content': f"{b['format_ok']}/{b['content_ok']}", 'after_format/content': f"{a['format_ok']}/{a['content_ok']}"} for b,a in zip(before_rows, after_rows)])
display(detail)

In [ ]:
result = {'config': CONFIG, 'versions': {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]}, 'method': 'LoRA（量子化なし）', 'train_size': len(train), 'valid_size': len(valid), 'before': before_summary, 'after': after_summary, 'regressed': regressed, 'executed_training_myself': True}
(WORK/'result.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(result, ensure_ascii=False, indent=2))
print('保存先:', WORK)